[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-crystal.ipynb)

# Full Project: Crystal Structure Classification

*AIBits Academy · Machine Learning End To End · Full Project*

A complete multi-class classification pipeline from materials science — cleaning messy scientific data, handling severe class imbalance, comparing seven model families, and tuning with validation curves.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['Crystal_structure.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> Materials scientists studying perovskite oxides (ABO₃ compounds, used in solid-state batteries, sensors, and catalysts) want to predict which of four crystal structures a new compound will adopt — **before** synthesising it in a lab. X-ray diffraction confirmation is slow and expensive; a model that predicts structure from cheap, tabulated elemental properties (ionic radius, electronegativity, bond length) can rank candidate compounds and dramatically cut the search space for materials discovery.

> **Dataset**
>
> **5,329 ABO₃ perovskite oxides, 15 features, 4-class target.** Features: element identity (A, B sites), oxidation states v(A)/v(B), ionic radii r(AXII)/r(AVI)/r(BVI), electronegativities EN(A)/EN(B), bond lengths l(A–O)/l(B–O), tolerance factors (t<sub>G</sub>, τ, μ). Target `Lowest distortion`: **cubic, orthorhombic, rhombohedral,** or **tetragonal**.

## Step 1 — Clean the Data

Real scientific exports are messy: missing values are encoded as literal `"-"` strings rather than NaN, and numeric columns are read in as text.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Crystal_structure.csv")
df.drop(columns=["In literature", "Compound"], inplace=True)

# Missing values are encoded as "-" strings, not NaN
df = df.replace("-", np.nan)

# Fix dtypes that got read in as object due to the "-" values
df["v(A)"] = pd.to_numeric(df["v(A)"], downcast="integer")
df["v(B)"] = pd.to_numeric(df["v(B)"], downcast="integer")
df["τ"]    = pd.to_numeric(df["τ"], downcast="float")

# Drop rows with no target label
df = df.dropna(subset=["Lowest distortion"])

# Frequency-weighted imputation for v(A)/v(B), grouped by class
for col in ["v(A)", "v(B)"]:
    grouped = df[df[col].notnull()].groupby("Lowest distortion")
    freq = grouped[col].apply(lambda x: x.value_counts())
    df[col] = df.apply(
        lambda row: np.random.choice(
            freq[row["Lowest distortion"]].index,
            p=freq[row["Lowest distortion"]].values / freq[row["Lowest distortion"]].values.sum()
        ) if np.isnan(row[col]) else row[col], axis=1
    )

# Median-fill the remaining sparse column
df["τ"].fillna(df["τ"].median(), inplace=True)
print(df.isna().sum().sum(), "missing values remaining")

## Step 2 — Remove Outlier Rows (IQR Method)

In [ ]:
from sklearn.preprocessing import StandardScaler

def detect_outlier_iqr(data):
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return [x for x in data if x < lower or x > upper]

cols_to_check = df.columns[4:-1]
outlier_counts = df[cols_to_check].apply(lambda row: len(detect_outlier_iqr(row)), axis=1)

# Drop rows with more than 2 outlying feature values
df = df[outlier_counts <= 2]
print(df.shape)

## Step 3 — The Class Imbalance Problem

In [ ]:
print(df["Lowest distortion"].value_counts(normalize=True).round(3) * 100)

Nearly two-thirds of compounds are cubic. A model that just predicts "cubic" every time would score 61.6% accuracy while being completely useless — this is the same trap covered on the Handling Imbalanced Data page, here in a genuine scientific dataset rather than a textbook example.

## Step 4 — Label Encoding and a Baseline Model

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

le = LabelEncoder()
le.fit(pd.concat([df['A'], df['B']]))
df['A'] = le.transform(df['A'])
df['B'] = le.transform(df['B'])

X = df.drop(columns=['Lowest distortion'])
y_le = LabelEncoder().fit(df['Lowest distortion'])   # xgboost >= 2 needs numeric class labels
y = pd.Series(y_le.transform(df['Lowest distortion']), index=df.index)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

baseline = xgb.XGBClassifier()
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
print("Baseline accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=y_le.classes_))

75.1% overall accuracy looks reasonable — but recall on **rhombohedral (0.07)** and **tetragonal (0.25)** is dismal. The model is essentially only good at the majority class, exactly the failure mode the class-balance chart predicted.

## Step 5 — Model Comparison with Oversampling

Before comparing models, the minority classes are oversampled in the training set only (never touch the test set) using `RandomOverSampler`, then seven model families are compared via 5-fold cross-validation:

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

oversample = RandomOverSampler(
    sampling_strategy={int(y_le.transform([k])[0]): v for k, v in {"orthorhombic": 1500, "rhombohedral": 750, "tetragonal": 500}.items()},
    random_state=42
)
X_train_os, y_train_os = oversample.fit_resample(X_train, y_train)

models = {
    "KNN": KNeighborsClassifier(), "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": xgb.XGBClassifier(), "SVM": SVC(),
    "RandomForest": RandomForestClassifier(), "GaussianNB": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(),
}
for name, model in models.items():
    cv = cross_validate(model, X_train_os, y_train_os, cv=5, return_train_score=True)
    print(f"{name:14s}  CV acc: {cv['test_score'].mean():.4f}   train acc: {cv['train_score'].mean():.4f}")

> **Reading Past the Headline Number**
>
> Random Forest's 89.6% CV accuracy looks like the clear winner — until you notice its train accuracy is a perfect 1.0. Both Random Forest and Decision Tree have **memorised the training set** (see Decision Trees for why unconstrained trees do this). XGBoost's smaller train/CV gap (0.773 vs 0.732, an 4.1-point gap, versus Random Forest's 10.4-point gap) signals a model that will generalise more reliably to genuinely new compounds — even though its raw CV number is lower. This is precisely why validation curves exist: a single accuracy number can't distinguish "good model" from "model that memorised the answer key."

## Step 6 — Validation Curves and Grid Search

In [ ]:
from yellowbrick.model_selection import validation_curve
from sklearn.model_selection import GridSearchCV

# Validation curve for max_depth (train vs CV score at each depth)
validation_curve(
    xgb.XGBClassifier(), X_train_os, y_train_os,
    param_name="max_depth", param_range=np.arange(1, 7, 1), cv=3, scoring="accuracy"
)
# -> training score climbs to 0.96 by depth 6 while CV score plateaus near 0.85:
#    the gap that starts opening past depth 4 is the overfitting signal.

param_grid = {
    "n_estimators": [70, 80, 85],
    "max_depth": [3, 4],
    "learning_rate": [0.08, 0.1, 0.12],
}
grid = GridSearchCV(xgb.XGBClassifier(), param_grid=param_grid, cv=2, n_jobs=2)
grid.fit(X_train_os, y_train_os)
print(grid.best_params_)

## Step 7 — Final Model: Before vs After

In [ ]:
final = xgb.XGBClassifier(n_estimators=85, max_depth=4, learning_rate=0.12)
final.fit(X_train_os, y_train_os)
y_pred = final.predict(X_test)
print("Final test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=y_le.classes_))

| Metric | Baseline (no oversampling, default params) | Final (oversampled + tuned) |
|---|---|---|
| Overall accuracy | 75.09% | 75.81% |
| Rhombohedral recall | 0.07 | **0.39** |
| Tetragonal recall | 0.25 | **0.45** |

The overall accuracy barely moved (+0.7 points) — but that number was always the wrong thing to optimise. Recall on the two rare-but-scientifically-interesting structures more than quintupled and nearly doubled respectively. For a materials-discovery use case where the rare structures are often the most novel and valuable candidates, this is the metric that actually matters.

## Visualizing the Class-Imbalance Fix

Precision / recall / F1 per class, toggled baseline vs. final. Watch rhombohedral and tetragonal recall specifically — the two rare classes the whole pipeline was built to rescue.

## Key Business Takeaways

- Real scientific/business data arrives messy — missing values encoded as literal strings, mixed dtypes — and cleaning is not optional preamble, it's most of the actual work.
- Feature selection (SelectKBest) was tried and made things worse here (74.3% vs 75.1% baseline) — a reminder that every technique needs to be validated on *this* dataset, not assumed to help by default.
- A lower cross-validation score with a smaller train/CV gap (XGBoost) is a more trustworthy choice than a higher score riding on visible overfitting (Random Forest, Decision Tree).
- Optimising overall accuracy under class imbalance can leave the metrics that matter most (minority-class recall) almost untouched; oversampling plus tuning must target those directly.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The accuracy of guessing the majority class

Store in `majority_share` the fraction of compounds in the most common structure class (`Lowest distortion`) - the accuracy of a model that always predicts it.

In [ ]:
majority_share = None   # TODO


In [ ]:
try:
    check("about 61.6%", abs(majority_share - 0.616) < 0.005)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
majority_share = df["Lowest distortion"].value_counts(normalize=True).max()

```

</details>

### Exercise 2 · Medium · Oversample only the training set

Apply `RandomOverSampler(random_state=0)` so **every class in the training data has as many rows as the majority class**. Store the resampled labels' class counts (dict) in `os_counts`. Never resample the test set.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
os_counts = None   # TODO (use X_train, y_train)


In [ ]:
try:
    check("four classes", len(os_counts) == 4)
    check("all equal", len(set(os_counts.values())) == 1)
    check("size = majority class", list(os_counts.values())[0] == y_train.value_counts().max())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
X_bal, y_bal = RandomOverSampler(random_state=0).fit_resample(X_train, y_train)
os_counts = dict(Counter(y_bal))

```

</details>

### Exercise 3 · Stretch · Per-class recall by hand

Using the baseline model, compute the recall of **each class** without `classification_report`: store a dict `recall_by_class` (keys are the encoded class numbers 0-3). Which class is worst?

In [ ]:
recall_by_class = {}   # TODO (predict with `baseline` on X_test)


In [ ]:
try:
    from sklearn.metrics import recall_score
    labels = sorted(set(np.asarray(y_test)))
    ref = dict(zip(labels, recall_score(y_test, baseline.predict(X_test), average=None, labels=labels)))
    check("same classes", set(recall_by_class) == set(ref))
    check("same values", all(abs(recall_by_class[k] - ref[k]) < 1e-9 for k in ref))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
yt = np.asarray(y_test); yp = np.asarray(baseline.predict(X_test))
recall_by_class = {int(c): float(((yp == c) & (yt == c)).sum() / (yt == c).sum()) for c in sorted(set(yt))}

```

Overall accuracy hides the rare classes; per-class recall exposes them.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Crystal Structure Classification**.*